# **Emotion Classification in Text Using TF-IDF and Machine Learning Models**

## Objective:
* Develop machine learning models to classify emotions in text samples.
* To preprocess textual data by performing cleaning, tokenization, and removal of irrelevant elements such as stopwords and special characters.
* To transform text data into numerical representations using TF-IDF vectorization for effective model training.
* To implement and train machine learning models, specifically Naive Bayes and Support Vector Machine (SVM), for emotion classification.
* To evaluate and compare the performance of the models using appropriate metrics such as accuracy and F1-score.
* To analyze the suitability of different models for multi-class emotion classification tasks based on their performance.

## Import Libraries

In [65]:
# Data Handling and Manipulation
import pandas as pd
import numpy as np

# Preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

# Model Building
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

# Model Evaluation
from sklearn.metrics import accuracy_score, classification_report

# 1. Loading and Preprocessing

## Data Understanding

In [66]:
# Load Dataset
df = pd.read_csv("/content/nlp_dataset.csv")
df

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear
...,...,...
5932,i begun to feel distressed for you,fear
5933,i left feeling annoyed and angry thinking that...,anger
5934,i were to ever get married i d have everything...,joy
5935,i feel reluctant in applying there because i w...,fear


In [67]:
# Basic Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Comment  5937 non-null   object
 1   Emotion  5937 non-null   object
dtypes: object(2)
memory usage: 92.9+ KB


In [68]:
# Check Missing Values
df.isnull().sum()

,0
Comment,0
Emotion,0


In [69]:
# Total Counts of ["Emotion"]
df["Emotion"].value_counts()

,count
Emotion,
anger,2000
joy,2000
fear,1937


## Data Preprocessing

In [70]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 3. Remove special characters & numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 4. Tokenization
    tokens = word_tokenize(text)

    # 5. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # 6. Join back
    return " ".join(tokens)

# Apply preprocessing
df['clean_comment'] = df['Comment'].apply(preprocess_text)

df[['Comment', 'clean_comment']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,Comment,clean_comment
0,i seriously hate one subject to death but now ...,seriously hate one subject death feel reluctan...
1,im so full of life i feel appalled,im full life feel appalled
2,i sit here to write i start to dig out my feel...,sit write start dig feelings think afraid acce...
3,ive been really angry with r and i feel like a...,ive really angry r feel like idiot trusting fi...
4,i feel suspicious if there is no one outside l...,feel suspicious one outside like rapture happe...


Text data was preprocessed by converting all text to lowercase, removing URLs, special characters, and numbers, and tokenizing the text into individual words. Stopwords were also removed to eliminate common but less meaningful words. Finally, the cleaned tokens were joined back into text for feature extraction.

These steps help reduce noise, standardize the data, and lower dimensionality, which improves model efficiency and performance. However, removing stopwords may sometimes affect meaning, especially in cases like negation (e.g., “not good”).

# 2. Feature Extraction

In [71]:
# Initialize TfidfVectorizer
tfidf = TfidfVectorizer()

Feature extraction was performed using **TF-IDF vectorization**, which **converts text into numerical vectors based on word importance rather than simple frequency**. TF-IDF assigns higher weights to words that are frequent in a document but rare across the dataset, making them more informative for classification. Compared to CountVectorizer, this approach reduces the influence of common words and improves model performance. However, both methods ignore word order and contextual meaning.

## Train Test Split

In [72]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['Emotion'],
    test_size=0.2,
    random_state=42
)

In [73]:
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

X_train : (4749, 7760)
X_test  : (1188, 7760)


# 3. Model Development

## 3.1 Train the Model

### a. Naive Bayers

In [74]:
# Train
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Predict
nb_y_pred = nb.predict(X_test)

# Evaluation
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, nb_y_pred)*100:.2f}%")

Naive Bayes Accuracy: 90.74%


### b. Support Vector Machine

In [75]:
# Train
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

# Predict
svm_y_pred = svm.predict(X_test)

# Evaluation
print(f"SVM Accuracy: {accuracy_score(y_test, svm_y_pred)*100:.2f}%")

SVM Accuracy: 94.70%


# 4. Model Comparison

In [89]:
# Naive Bayes
print("Naive Bayes: ")
print("-----------")
print(classification_report(y_test, nb_y_pred))
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, nb_y_pred)*100:.2f}%\n")

# Support Vector Machine
print("SVM: ")
print("----")
print(classification_report(y_test, svm_y_pred))
print(f"SVM Accuracy: {accuracy_score(y_test, svm_y_pred)*100:.2f}%")

Naive Bayes: 
-----------
              precision    recall  f1-score   support

       anger       0.88      0.94      0.91       392
        fear       0.92      0.92      0.92       416
         joy       0.93      0.86      0.90       380

    accuracy                           0.91      1188
   macro avg       0.91      0.91      0.91      1188
weighted avg       0.91      0.91      0.91      1188

Naive Bayes Accuracy: 90.74%

SVM: 
----
              precision    recall  f1-score   support

       anger       0.93      0.96      0.94       392
        fear       0.97      0.91      0.94       416
         joy       0.95      0.97      0.96       380

    accuracy                           0.95      1188
   macro avg       0.95      0.95      0.95      1188
weighted avg       0.95      0.95      0.95      1188

SVM Accuracy: 94.70%


# Model Explanation and Suitability

## Naive Bayes
The Multinomial Naive Bayes model is a probabilistic classifier that applies Bayes’ theorem with the assumption that features are conditionally independent. It is particularly effective for text classification tasks because it handles high-dimensional sparse data efficiently and requires less computational power. In this study, Naive Bayes achieved an accuracy of approximately 90.7% with balanced F1-scores across all emotion classes. However, its independence assumption limits its ability to capture relationships between words, which can reduce performance in more complex language patterns.

## SVM
The Support Vector Machine (SVM) model is a discriminative classifier that identifies an optimal hyperplane to separate different classes in a high-dimensional feature space. Using a linear kernel, SVM performs well with TF-IDF features due to their sparsity and high dimensionality. In this study, SVM achieved a higher accuracy of approximately 94.7% and superior F1-scores across all emotion categories, indicating better generalization and class separation compared to Naive Bayes.

## Conclusion:
Comparing both models, SVM outperforms Naive Bayes in terms of accuracy and F1-score, making it more suitable for emotion classification tasks. This is because SVM can better capture complex decision boundaries in high-dimensional text data, whereas Naive Bayes relies on a simplifying independence assumption. Therefore, SVM is the preferred model for this task due to its higher predictive performance and robustness.